In [27]:
import duckdb
import pandas as pd

In [28]:
con= duckdb.connect(database='dados_duckdb.db', read_only=False)

In [29]:
df = con.execute('SELECT * FROM BRONZE_Z0019').fetch_df()
df

,NATBR,MAKYX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-10-31 21:58:34.426433
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-10-31 21:58:34.426433
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-10-31 21:58:34.426433
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-02 09:58:51.973377
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-02 09:58:51.973377
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-02 09:58:51.973377


In [30]:
df = con.execute("""
SELECT * FROM (
    SELECT * , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) as row
    FROM BRONZE_Z0019
    WHERE data_ingestao>= '2026-10-31'
    ) WHERE row=1
""").fetch_df()
df

,NATBR,MAKYX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-10-31 21:58:34.426433,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-10-31 21:58:34.426433,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-02 09:58:51.973377,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-02 09:58:51.973377,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-02 09:58:51.973377,1


In [36]:
df_final= df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKYX":"produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_produto"})
df_final

,id,produto,id_categoria,id_fornecedor,vl_produto
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [37]:
df_final.dtypes

id               object
produto          object
id_categoria     object
id_fornecedor    object
vl_produto       object
dtype: object

In [47]:
df2 = df_final
df2 = df_final.astype(
    {
        'id':int,
        "produto": str,
        "id_categoria":str,
        "id_fornecedor":int,
        "vl_produto":float }
)

In [48]:
df2.dtypes

id                 int32
produto           object
id_categoria      object
id_fornecedor      int32
vl_produto       float64
dtype: object

In [64]:
con.execute(""" CREATE TABLE IF NOT EXISTS PRODUTOS(
    id BIGINT,
    produto TEXT,
    id_categoria TEXT,
    id_fornecedor BIGINT,
    vl_produto FLOAT     )
""")

In [ ]:
# con.execute("drop table produtos")

In [49]:
df2

,id,produto,id_categoria,id_fornecedor,vl_produto
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [70]:
df_resultado = con.execute("select * from produtos").fetch_df()
df_resultado

,id,produto,id_categoria,id_fornecedor,vl_produto
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [65]:
con.execute("insert into produtos select * from df2")

In [68]:
df2.dtypes

id                 int32
produto           object
id_categoria      object
id_fornecedor      int32
vl_produto       float64
dtype: object

In [69]:
df2

,id,produto,id_categoria,id_fornecedor,vl_produto
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [71]:
con.close()